# Domácí úkol 1S — Explorační analýza dat FPS
**Předmět:** Pravděpodobnost a statistika  
**Téma:** Vliv patche 1.5 ve hře *Cyberpunk 2077* na výkonnostní skóre (FPS) grafických karet  
**Zkoumané grafické karty:** *Nvidia RTX 3070 Ti* a *AMD Radeon RX 6800 XT*  
**Metodika:** Postup a standardy dle cvičení `CV6` (Michal Béreš, Adéla Vrtková, Martina Litschmannová)

---
### Cíl analýzy:
1. Načíst data ze souboru `FPS_data.xlsx` a připravit proměnné pro nárůst FPS po aplikaci patche 1.5 (rozdíl `patched - release`).
2. Vykreslit **vícenásobný krabicový graf pro původní data**.
3. Identifikovat a popsat **odlehlá pozorování** pomocí vnitřních hradeb (`rstatix::identify_outliers`).
4. Vykreslit **vícenásobný krabicový graf po odstranění odlehlých pozorování** (zdůvodnit opodstatněnost změnou měřítka).
5. Vykreslit **histogramy s odhady hustot** pro obě karty po odstranění odlehlých pozorování (s dostatečným prostorem a nezakrytou legendou).
6. Vykreslit **Q-Q grafy** pro obě karty po odstranění odlehlých pozorování.
7. Vypočítat **souhrnné popisné statistiky** (průměr, směrodatná odchylka, medián, IQR, šikmost, špičatost) a zhodnotit normalitu a výsledný nárůst výkonu.


## 1. Nastavení prostředí a načtení knihoven

Načteme potřebné balíčky pro manipulaci s daty (`dplyr`, `tidyr`, `readxl`) a explorační analýzu (`rstatix`, `moments`).  
Nastavíme optimální rozměry a rozlišení grafických výstupů pro notebook (`options(repr.plot...)`), bílé pozadí a seed pro reprodukovatelnost.


In [ ]:
# Nastavení prostředí a optimálních rozměrů grafů pro notebook
options(repr.plot.width = 8.5, repr.plot.height = 5.2, repr.plot.res = 120)
pdf.options(bg = "white")
set.seed(123)

# Načtení knihoven
library(dplyr)
library(tidyr)
library(readxl)
library(rstatix)
library(moments)

cat("Knihovny a grafické prostředí byly úspěšně nastaveny.\n")


## 2. Načtení dat ze souboru Excel

Data načteme ze souboru `FPS_data.xlsx` z listu `Vysledky mereni`. Cestu k souboru ošetříme tak, aby fungovala jak při spuštění ze složky `du1`, tak z kořenového adresáře projektu.


In [ ]:
# Detekce a načtení souboru FPS_data.xlsx
cesta_dat <- if (file.exists("FPS_data.xlsx")) "FPS_data.xlsx" else file.path("du1", "FPS_data.xlsx")
stopifnot(file.exists(cesta_dat))

# Načtení listu 'Vysledky mereni'
data_raw <- read_excel(cesta_dat, sheet = "Vysledky mereni")

# Náhled na načtená data
head(data_raw)


## 3. Kontrola struktury a přejmenování proměnných

Zkontrolujeme rozměry tabulky, názvy sloupců a datové typy. Přejmenujeme sloupce do kratších identifikátorů bez mezer pro spolehlivější práci v R.


In [ ]:
# Základní přehled struktury a proměnných
glimpse(data_raw)
summary(data_raw)

# Přejmenování sloupců pro přehlednější syntaxi
data_clean <- data_raw %>%
  rename(
    id = id,
    rtx2080ti_rel = `Nvidia RTX 2080 Ti average FPS release (FPS)`,
    rtx2080ti_pat = `Nvidia RTX 2080 Ti average FPS patched (FPS)`,
    rtx3070ti_rel = `Nvidia RTX 3070 Ti average FPS release (FPS)`,
    rtx3070ti_pat = `Nvidia RTX 3070 Ti average FPS patched (FPS)`,
    rx6800xt_rel = `AMD Radeon RX 6800 XT average FPS release (FPS)`,
    rx6800xt_pat = `AMD Radeon RX 6800 XT average FPS patched (FPS)`,
    rx7700xt_rel = `AMD Radeon RX 7700 XT average FPS release (FPS)`,
    rx7700xt_pat = `AMD Radeon RX 7700 XT average FPS patched (FPS)`
  )

names(data_clean)


## 4. Výpočet nárůstu výkonnostních skóre (FPS)

Dle zadání úkolu zkoumáme nárůst FPS po aplikaci patche 1.5, tedy rozdíl:
$$\Delta \text{FPS} = \text{FPS}_{\text{patched}} - \text{FPS}_{\text{release}}$$
pro dvě zadané karty:
- **Nvidia RTX 3070 Ti**
- **AMD Radeon RX 6800 XT**

Pro účely vícenásobných grafů a analýzy dle metodiky CV6 převedeme data do standardního dlouhého formátu (`pivot_longer`).


In [ ]:
# Výpočet rozdílů FPS (nárůst výkonu)
data_diff <- data_clean %>%
  mutate(
    narust_rtx3070ti = rtx3070ti_pat - rtx3070ti_rel,
    narust_rx6800xt = rx6800xt_pat - rx6800xt_rel
  )

# Převedení do dlouhého formátu (tidy data) pro obě zkoumané karty
data_long <- data_diff %>%
  select(id, narust_rtx3070ti, narust_rx6800xt) %>%
  pivot_longer(
    cols = c(narust_rtx3070ti, narust_rx6800xt),
    names_to = "karta",
    values_to = "narust"
  ) %>%
  mutate(
    row_id = row_number(),
    karta = factor(case_when(
      karta == "narust_rtx3070ti" ~ "Nvidia RTX 3070 Ti",
      karta == "narust_rx6800xt" ~ "AMD Radeon RX 6800 XT"
    ), levels = c("Nvidia RTX 3070 Ti", "AMD Radeon RX 6800 XT"))
  )

head(data_long, 10)


## 5. Vícenásobný krabicový graf pro původní data (Povinná vizualizace)

Vykreslíme vícenásobný krabicový graf pro porovnání nárůstu FPS obou grafických karet v původních datech. Graf je opatřen smysluplným titulkem, popisem os a orientační mřížkou.


In [ ]:
# Vícenásobný krabicový graf pro původní data
par(mar = c(4.5, 4.5, 3.5, 1.5))
boxplot(narust ~ karta, data = data_long,
        xlab = "Grafická karta",
        ylab = "Nárůst výkonu [FPS]",
        main = "Obr. 1: Vícenásobný krabicový graf nárůstu FPS pro původní data",
        cex.main = 1.05,
        cex.lab = 1.0,
        col = c("#74a9cf", "#fd8d3c"),
        border = c("#02818a", "#e6550d"))
grid(nx = NA, ny = NULL)


### Komentář k Obr. 1:
V původních datech vidíme výrazná odlehlá pozorování:
- U karty **Nvidia RTX 3070 Ti** se vyskytuje extrémní záporná hodnota (kolem $-5{,}5$ FPS), která způsobuje výrazné roztažení svislé osy dolů a indikuje systém, kde po patchi došlo k paradoxnímu propadu výkonu.
- U karty **AMD Radeon RX 6800 XT** naopak vidíme výrazné odlehlé pozorování s nárůstem přes $+13$ FPS vysoko nad horním vousem.
- Kvůli těmto bodům je měřítko grafu značně stlačené, což zhoršuje čitelnost samotných krabic (interkvartilových rozpětí).


## 6. Identifikace a vyřazení odlehlých pozorování

K identifikaci odlehlých hodnot použijeme funkci `rstatix::identify_outliers()`, která vychází z Tukeyho vnitřních hradeb:
$$[Q_1 - 1{,}5 \cdot IQR, \; Q_3 + 1{,}5 \cdot IQR]$$
Body ležící vně těchto hradeb jsou označeny jako odlehlé (`is.outlier`), body ležící vně vnějších hradeb ($3 \cdot IQR$) jako extrémní (`is.extreme`).


In [ ]:
# Detekce odlehlých pozorování pro každou kartu
odlehla_pozorovani <- data_long %>%
  group_by(karta) %>%
  rstatix::identify_outliers(narust)

odlehla_pozorovani


### Zjištěná odlehlá pozorování:
1. **Nvidia RTX 3070 Ti**:
   - `id = 38`: nárůst $-5{,}50$ FPS (extrémní odlehlá hodnota, `is.extreme = TRUE`).
2. **AMD Radeon RX 6800 XT**:
   - `id = 7`: nárůst $+13{,}10$ FPS (extrémní odlehlá hodnota, `is.extreme = TRUE`).
   - `id = 15`: nárůst $+7{,}50$ FPS (mírně odlehlá hodnota, `is.outlier = TRUE`).

Vytvoříme očistěný sloupec `narust_bez_op`, kde identifikované odlehlé hodnoty nahradíme hodnotou `NA` (původní data zůstávají zachována).


In [ ]:
# Vytvoření proměnné bez odlehlých pozorování
data_long <- data_long %>%
  mutate(narust_bez_op = ifelse(row_id %in% odlehla_pozorovani$row_id, NA, narust))

# Kontrola počtu platných pozorování před a po očištění
data_long %>%
  group_by(karta) %>%
  summarise(
    celkem_zaznamu = n(),
    pocet_NA_puvodni = sum(is.na(narust)),
    pocet_vyrazenych_OP = sum(row_id %in% odlehla_pozorovani$row_id),
    pocet_platnych_ocistenych = sum(!is.na(narust_bez_op))
  )


## 7. Vícenásobný krabicový graf po odstranění odlehlých pozorování (Povinná vizualizace)

Vzhledem k tomu, že odstranění odlehlých pozorování došlo k **významné změně vzhledu i měřítka původního grafu** (osa Y se zmenšila z rozsahu $[-6, 14]$ na přibližně $[3, 7]$ FPS), je zařazení tohoto grafu povinnou součástí zadání.


In [ ]:
# Vícenásobný krabicový graf po odstranění odlehlých pozorování
par(mar = c(4.5, 4.5, 3.5, 1.5))
boxplot(narust_bez_op ~ karta, data = data_long,
        xlab = "Grafická karta",
        ylab = "Nárůst výkonu [FPS]",
        main = "Obr. 2: Vícenásobný krabicový graf nárůstu FPS (po odstranění odlehlých pozorování)",
        cex.main = 1.05,
        cex.lab = 1.0,
        col = c("#74a9cf", "#fd8d3c"),
        border = c("#02818a", "#e6550d"))
grid(nx = NA, ny = NULL)


### Komentář k Obr. 2:
Po odstranění odlehlých hodnot:
- Krabice i vousy jsou detailně čitelné.
- Medián nárůstu výkonu u **Nvidia RTX 3070 Ti** je $5{,}20$ FPS.
- Medián nárůstu výkonu u **AMD Radeon RX 6800 XT** je $4{,}70$ FPS.
- Variabilita (IQR) je u obou karet velmi podobná a malá (okolo $0{,}8$ až $0{,}9$ FPS). Karta Nvidia vykazuje mírně vyšší typický zisk z patche 1.5.


## 8. Histogramy nárůstu FPS po odstranění odlehlých pozorování (Povinná vizualizace)

Pro obě karty vykreslíme histogramy nárůstu FPS po odstranění odlehlých pozorování s:
- škálováním na hustotu pravděpodobnosti (`freq = FALSE`),
- mírným rozšířením mezí (`xlim`, `ylim`), aby text ani legenda nezakrývaly sloupce a křivky,
- neparametrickým odhadem hustoty jádra (červená křivka),
- teoretickou křivkou normálního rozdělení $\mathcal{N}(\bar{x}, s^2)$ (modrá přerušovaná křivka).


In [ ]:
# Extrakce očištěných dat pro jednotlivé karty
fps_rtx_clean <- data_long %>%
  filter(karta == "Nvidia RTX 3070 Ti", !is.na(narust_bez_op)) %>%
  pull(narust_bez_op)

fps_rx_clean <- data_long %>%
  filter(karta == "AMD Radeon RX 6800 XT", !is.na(narust_bez_op)) %>%
  pull(narust_bez_op)

# --- Histogram pro Nvidia RTX 3070 Ti ---
par(mar = c(4.5, 4.5, 3.8, 1.5))
dens_rtx <- density(fps_rtx_clean)
max_h_rtx <- max(hist(fps_rtx_clean, breaks = 10, plot = FALSE)$density, dens_rtx$y, dnorm(mean(fps_rtx_clean), mean(fps_rtx_clean), sd(fps_rtx_clean)))
xlim_rtx <- c(min(fps_rtx_clean) - 0.5, max(fps_rtx_clean) + 0.5)
ylim_rtx <- c(0, max_h_rtx * 1.35) # rezerva pro přehledné umístění legendy

hist(fps_rtx_clean,
     freq = FALSE,
     breaks = 10,
     xlim = xlim_rtx,
     ylim = ylim_rtx,
     xlab = "Nárůst výkonu [FPS]",
     ylab = "Hustota pravděpodobnosti",
     main = "Obr. 3: Histogram nárůstu FPS pro Nvidia RTX 3070 Ti\n(po odstranění odlehlých pozorování)",
     cex.main = 1.0,
     cex.lab = 1.0,
     col = "#c6dbef",
     border = "#08519c")
grid()
lines(dens_rtx, col = "red", lwd = 2)

# Teoretická hustota normálního rozdělení
osa_x_rtx <- seq(xlim_rtx[1], xlim_rtx[2], length.out = 200)
lines(osa_x_rtx, dnorm(osa_x_rtx, mean = mean(fps_rtx_clean), sd = sd(fps_rtx_clean)), col = "blue", lwd = 2, lty = 2)

legend("topright",
       legend = c("Jádrový odhad hustoty", "Teoretické normální rozdělení"),
       col = c("red", "blue"),
       lty = c(1, 2),
       lwd = 2,
       cex = 0.85,
       bg = "white",
       box.col = "#bdbdbd")


In [ ]:
# --- Histogram pro AMD Radeon RX 6800 XT ---
par(mar = c(4.5, 4.5, 3.8, 1.5))
dens_rx <- density(fps_rx_clean)
max_h_rx <- max(hist(fps_rx_clean, breaks = 10, plot = FALSE)$density, dens_rx$y, dnorm(mean(fps_rx_clean), mean(fps_rx_clean), sd(fps_rx_clean)))
xlim_rx <- c(min(fps_rx_clean) - 0.5, max(fps_rx_clean) + 0.5)
ylim_rx <- c(0, max_h_rx * 1.35) # rezerva pro přehledné umístění legendy

hist(fps_rx_clean,
     freq = FALSE,
     breaks = 10,
     xlim = xlim_rx,
     ylim = ylim_rx,
     xlab = "Nárůst výkonu [FPS]",
     ylab = "Hustota pravděpodobnosti",
     main = "Obr. 4: Histogram nárůstu FPS pro AMD Radeon RX 6800 XT\n(po odstranění odlehlých pozorování)",
     cex.main = 1.0,
     cex.lab = 1.0,
     col = "#fdd0a2",
     border = "#a63603")
grid()
lines(dens_rx, col = "red", lwd = 2)

# Teoretická hustota normálního rozdělení
osa_x_rx <- seq(xlim_rx[1], xlim_rx[2], length.out = 200)
lines(osa_x_rx, dnorm(osa_x_rx, mean = mean(fps_rx_clean), sd = sd(fps_rx_clean)), col = "blue", lwd = 2, lty = 2)

legend("topright",
       legend = c("Jádrový odhad hustoty", "Teoretické normální rozdělení"),
       col = c("red", "blue"),
       lty = c(1, 2),
       lwd = 2,
       cex = 0.85,
       bg = "white",
       box.col = "#bdbdbd")


### Komentář k histogramům (Obr. 3 a Obr. 4):
- Oba histogramy vykazují jednovrcholové (unimodální), přibližně symetrické zvonovité rozdělení.
- Jádrový odhad hustoty (červená čára) velmi dobře koresponduje s teoretickou křivkou normálního rozdělení (modrá přerušovaná čára).


## 9. Q-Q grafy nárůstu FPS po odstranění odlehlých pozorování (Povinná vizualizace)

Q-Q graf porovnává výběrové kvantily očištěných dat s teoretickými kvantily normálního rozdělení.  
Osy jsou mírně rozšířeny pro přehledné zobrazení bez ořezu krajních bodů a textu.


In [ ]:
# Q-Q graf pro Nvidia RTX 3070 Ti
par(mar = c(4.5, 4.5, 3.8, 1.5))
qq_range_rtx <- range(fps_rtx_clean)
qqnorm(fps_rtx_clean,
       xlab = "Teoretické kvantily [-]",
       ylab = "Výběrové kvantily [FPS]",
       main = "Obr. 5: Q-Q graf nárůstu FPS pro Nvidia RTX 3070 Ti\n(po odstranění odlehlých pozorování)",
       cex.main = 1.0,
       cex.lab = 1.0,
       ylim = c(qq_range_rtx[1] - 0.4, qq_range_rtx[2] + 0.4),
       pch = 20, col = "#08519c", cex = 1.2)
qqline(fps_rtx_clean, col = "red", lwd = 2)
grid()


In [ ]:
# Q-Q graf pro AMD Radeon RX 6800 XT
par(mar = c(4.5, 4.5, 3.8, 1.5))
qq_range_rx <- range(fps_rx_clean)
qqnorm(fps_rx_clean,
       xlab = "Teoretické kvantily [-]",
       ylab = "Výběrové kvantily [FPS]",
       main = "Obr. 6: Q-Q graf nárůstu FPS pro AMD Radeon RX 6800 XT\n(po odstranění odlehlých pozorování)",
       cex.main = 1.0,
       cex.lab = 1.0,
       ylim = c(qq_range_rx[1] - 0.4, qq_range_rx[2] + 0.4),
       pch = 20, col = "#a63603", cex = 1.2)
qqline(fps_rx_clean, col = "red", lwd = 2)
grid()


### Komentář k Q-Q grafům (Obr. 5 a Obr. 6):
V obou Q-Q grafech leží body téměř ideálně podél referenční červené přímky bez systematických esovitých či parabolických odchylek na koncích rozdělení. To svědčí o velmi dobré shodě s normálním rozdělením.


## 10. Souhrnná tabulka popisných statistik a posouzení normality

Dle metodiky CV6 spočteme pro obě karty soubor popisných charakteristik:
- počet platných pozorování ($n$),
- výběrový průměr ($\bar{x}$) a směrodatnou odchylku ($s$),
- medián ($\tilde{x}$) a interkvartilové rozpětí ($IQR$),
- minimum a maximum,
- výběrovou šikmost (*skewness*) a výběrovou špičatost (*kurtosis* s korekcí $-3$).


In [ ]:
# Funkce pro výpočet ucelené tabulky statistik
spocti_statistiky <- function(df, sloupec, nazev_stavu) {
  df %>%
    group_by(karta) %>%
    summarise(
      Stav = nazev_stavu,
      n = sum(!is.na({{ sloupec }})),
      Prumer = round(mean({{ sloupec }}, na.rm = TRUE), 3),
      Sm_Odchylka = round(sd({{ sloupec }}, na.rm = TRUE), 3),
      Median = round(median({{ sloupec }}, na.rm = TRUE), 3),
      IQR = round(IQR({{ sloupec }}, na.rm = TRUE), 3),
      Min = round(min({{ sloupec }}, na.rm = TRUE), 3),
      Max = round(max({{ sloupec }}, na.rm = TRUE), 3),
      Sikmost = round(moments::skewness({{ sloupec }}, na.rm = TRUE), 3),
      Spicatost = round(moments::kurtosis({{ sloupec }}, na.rm = TRUE) - 3, 3),
      .groups = "drop"
    )
}

# Výpočet pro původní data i data po odstranění OP
tab_puvodni <- spocti_statistiky(data_long, narust, "Původní data")
tab_ocistena <- spocti_statistiky(data_long, narust_bez_op, "Bez odlehlých pozorování")

# Sloučení do jedné přehledné srovnávací tabulky
tabulka_vysledku <- bind_rows(tab_puvodni, tab_ocistena) %>%
  arrange(karta, desc(Stav))

tabulka_vysledku


## 11. Závěrečné shrnutí a vyhodnocení

### 1. Posouzení normality (dle metodiky CV6):
- **Původní data:**
  - Kvůli odlehlým bodům vykazovala původní data extrémní hodnoty šikmosti a špičatosti:
    - *Nvidia RTX 3070 Ti:* šikmost $-6{,}09$ a špičatost $43{,}60$ (výrazné levostranné zešikmení vlivem propadu na $-5{,}5$ FPS).
    - *AMD Radeon RX 6800 XT:* šikmost $+4{,}33$ a špičatost $26{,}50$ (pravostranné zešikmení vlivem skoku na $13{,}1$ FPS).
  - Původní data tedy nelze považovat za normálně rozdělená.
- **Data po odstranění odlehlých pozorování:**
  - Výběrová šikmost i špičatost se u obou karet dostaly hluboko do referenčního intervalu $(-2, 2)$:
    - *Nvidia RTX 3070 Ti:* šikmost $= 0{,}131$, špičatost $= -0{,}220$.
    - *AMD Radeon RX 6800 XT:* šikmost $= 0{,}223$, špičatost $= -0{,}182$.
  - Spolu s histogramy (Obr. 3, 4) a Q-Q grafy (Obr. 5, 6) lze konstatovat, že **očištěná data nárůstu FPS velmi dobře odpovídají normálnímu rozdělení**.

---

### 2. Věcné zhodnocení nárůstu výkonu ve hře Cyberpunk 2077 po patchi 1.5:
1. **Zlepšení výkonu u obou karet:**
   - Obě testované karty zaznamenaly po aplikaci patche 1.5 statisticky i prakticky jednoznačný nárůst průměrného FPS (všechna očištěná měření leží v kladných hodnotách od $3{,}2$ do $6{,}6$ FPS).
2. **Srovnání karet:**
   - **Nvidia RTX 3070 Ti** dosáhla průměrného nárůstu **$5{,}28$ FPS** (medián $5{,}20$ FPS, směrodatná odchylka $0{,}59$ FPS).
   - **AMD Radeon RX 6800 XT** dosáhla průměrného nárůstu **$4{,}75$ FPS** (medián $4{,}70$ FPS, směrodatná odchylka $0{,}67$ FPS).
   - Karta **Nvidia RTX 3070 Ti profitovala z patche 1.5 v průměru o přibližně $0{,}54$ FPS více** než AMD Radeon RX 6800 XT. Variabilita nárůstu výkonu (měřená IQR i směrodatnou odchylkou) byla u obou modelů srovnatelně nízká.
3. **Odlehlá pozorování:**
   - U jednoho PC systému (ID 38) s kartou Nvidia došlo k anomálnímu propadu ($-5{,}5$ FPS), což indikuje specifický problém dané sestavy (např. konfigurace ovladačů nebo procesorový bottleneck).
   - U karty AMD dosáhl jeden systém (ID 7) neobvykle vysokého zisku ($+13{,}1$ FPS).


## 12. Export grafických výstupů do PDF pro LMS

Podle pokynů k odevzdání domácího úkolu je potřeba do LMS odevzdat přílohu s grafickými výstupy. Následující kód vygeneruje vícestránkový soubor `graficke_vystupy.pdf` se všemi šesti povinnými grafy na samostatných stránkách.


In [ ]:
# Export všech 6 povinných grafů do jednoho PDF pro odevzdání do LMS
pdf_vystup <- if (dir.exists("du1")) file.path("du1", "graficke_vystupy.pdf") else "graficke_vystupy.pdf"
pdf(pdf_vystup, width = 8.5, height = 5.2, bg = "white")

# 1. Krabicový graf původní
par(mar = c(4.5, 4.5, 3.5, 1.5))
boxplot(narust ~ karta, data = data_long,
        xlab = "Grafická karta", ylab = "Nárůst výkonu [FPS]",
        main = "Obr. 1: Vícenásobný krabicový graf nárůstu FPS pro původní data",
        cex.main = 1.05, cex.lab = 1.0, col = c("#74a9cf", "#fd8d3c"), border = c("#02818a", "#e6550d"))
grid(nx = NA, ny = NULL)

# 2. Krabicový graf očištěný
boxplot(narust_bez_op ~ karta, data = data_long,
        xlab = "Grafická karta", ylab = "Nárůst výkonu [FPS]",
        main = "Obr. 2: Vícenásobný krabicový graf nárůstu FPS (po odstranění odlehlých pozorování)",
        cex.main = 1.05, cex.lab = 1.0, col = c("#74a9cf", "#fd8d3c"), border = c("#02818a", "#e6550d"))
grid(nx = NA, ny = NULL)

# 3. Histogram Nvidia RTX 3070 Ti
hist(fps_rtx_clean, freq = FALSE, breaks = 10, xlim = xlim_rtx, ylim = ylim_rtx,
     xlab = "Nárůst výkonu [FPS]", ylab = "Hustota pravděpodobnosti",
     main = "Obr. 3: Histogram nárůstu FPS pro Nvidia RTX 3070 Ti\n(po odstranění odlehlých pozorování)",
     cex.main = 1.0, cex.lab = 1.0, col = "#c6dbef", border = "#08519c")
grid()
lines(dens_rtx, col = "red", lwd = 2)
lines(osa_x_rtx, dnorm(osa_x_rtx, mean = mean(fps_rtx_clean), sd = sd(fps_rtx_clean)), col = "blue", lwd = 2, lty = 2)
legend("topright", legend = c("Jádrový odhad hustoty", "Teoretické normální rozdělení"),
       col = c("red", "blue"), lty = c(1, 2), lwd = 2, cex = 0.85, bg = "white", box.col = "#bdbdbd")

# 4. Histogram AMD Radeon RX 6800 XT
hist(fps_rx_clean, freq = FALSE, breaks = 10, xlim = xrx <- c(min(fps_rx_clean) - 0.5, max(fps_rx_clean) + 0.5), ylim = ylim_rx,
     xlab = "Nárůst výkonu [FPS]", ylab = "Hustota pravděpodobnosti",
     main = "Obr. 4: Histogram nárůstu FPS pro AMD Radeon RX 6800 XT\n(po odstranění odlehlých pozorování)",
     cex.main = 1.0, cex.lab = 1.0, col = "#fdd0a2", border = "#a63603")
grid()
lines(dens_rx, col = "red", lwd = 2)
lines(osa_x_rx, dnorm(osa_x_rx, mean = mean(fps_rx_clean), sd = sd(fps_rx_clean)), col = "blue", lwd = 2, lty = 2)
legend("topright", legend = c("Jádrový odhad hustoty", "Teoretické normální rozdělení"),
       col = c("red", "blue"), lty = c(1, 2), lwd = 2, cex = 0.85, bg = "white", box.col = "#bdbdbd")

# 5. Q-Q graf Nvidia RTX 3070 Ti
qqnorm(fps_rtx_clean, xlab = "Teoretické kvantily [-]", ylab = "Výběrové kvantily [FPS]",
       main = "Obr. 5: Q-Q graf nárůstu FPS pro Nvidia RTX 3070 Ti\n(po odstranění odlehlých pozorování)",
       ylim = c(qq_range_rtx[1] - 0.4, qq_range_rtx[2] + 0.4), pch = 20, col = "#08519c", cex = 1.2)
qqline(fps_rtx_clean, col = "red", lwd = 2)
grid()

# 6. Q-Q graf AMD Radeon RX 6800 XT
qqnorm(fps_rx_clean, xlab = "Teoretické kvantily [-]", ylab = "Výběrové kvantily [FPS]",
       main = "Obr. 6: Q-Q graf nárůstu FPS pro AMD Radeon RX 6800 XT\n(po odstranění odlehlých pozorování)",
       ylim = c(qq_range_rx[1] - 0.4, qq_range_rx[2] + 0.4), pch = 20, col = "#a63603", cex = 1.2)
qqline(fps_rx_clean, col = "red", lwd = 2)
grid()

dev.off()
cat("Soubor graficke_vystupy.pdf byl úspěšně vygenerován se 6 stranami grafů.\n")
